# Step 03: Infer first and last corresponding author gender

Input:
    data/raw/gender_classification/预测国家列表.xlsx
    data/raw/gender_classification/训练样本-国家/code_{co}.csv
    
    data/interim/publication/2_first_bycountry/code_{co2}.csv
    data/interim/publication/2_last_bycountry/code_{co2}.csv


Output:
    data/interim/publication/2.1_first_bycountry_predicted/pred_{co2}.csv
    data/check/gender_classification/1.1_first_预测准确率.csv
    data/interim/publication/2_first_combined_predictresult.csv


    data/interim/publication/2.1_last_bycountry_predicted/pred_{co2}.csv
    data/check/gender_classification/1.1_last_预测准确率.csv
    data/interim/publication/2_last_combined_predictresult.csv

Purpose:
    - Predicted authors' gender

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
#忽略警告
import warnings
from tqdm import tqdm

In [ ]:
# -----------------------------
# Paths
# -----------------------------

PROJECT_ROOT = Path(__file__).resolve().parents[2]
FIRST_IN_FILE = PROJECT_ROOT / "data" / "interim" / "publication" / "2_first_bycountry"/ "code_{co}.csv"

FILE_PATH = PROJECT_ROOT / "data" / "raw" / "gender_classification" / "预测国家列表.xlsx"
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "gender_classification" / "训练样本-国家" / "code_{co}.csv"
TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)

FIRST_OUT_PATH = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_first_bycountry_predicted" / "pred_{co2}.csv"
FIRST_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

FIRST_ALL_OUT = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_first_NameAndCountry_Predicted.csv"
FIRST_ALL_OUT.parent.mkdir(parents=True, exist_ok=True)

FIRST_CHECK_OUT = PROJECT_ROOT / "data" / "check" / "gender_classification" / "1.1_first_预测准确率.csv"
FIRST_CHECK_OUT.parent.mkdir(parents=True, exist_ok=True)

# 1 First authors

In [ ]:
#warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore')
# 定义概率阈值
confidence_level = 0.7

# 读取数据
data = pd.read_excel(
   FILE_PATH
   )
data2 = pd.DataFrame(columns=['样本', '预测', '准确率'])
data

In [ ]:
# 定义预处理函数
def preprocessor(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub('[^a-zA-Z]', '', x)
    return x.lower()



# 创建向量化器和分类器
vec = CountVectorizer(
    min_df=50,
    ngram_range=(2, 8),
    analyzer='char',
    preprocessor=lambda x: f'^{x.lower().replace(" ", "")},'
)

clf = LogisticRegression(max_iter=1000)
pipeline = Pipeline([('vectorizer', vec), ('logreg', clf)])



def predict_gender(names):
    probas = pipeline.predict_proba(names)
    genders = []
    max_probas = []
    for proba in probas:
        if proba[0] > confidence_level:
            genders.append('F')
            max_probas.append(proba[0])
        elif proba[1] > confidence_level:
            genders.append('M')
            max_probas.append(proba[0])
        else:
            genders.append('')
            max_probas.append(proba[0])
    return pd.DataFrame({'genders': genders, 'probability': max_probas})

In [ ]:
# 循环处理每对样本和预测
# train_path为各个国家训练集所在的文件，格式为code_国家名.csv
for index, row in data.iterrows():    

    #if index>10:
        #continue
        
    co = row['样本']
    co2 = row['预测']

    print(f"Processing: Sample - {co}, Prediction - {co2}")

    train_path = TRAIN_PATH
    file_path = FIRST_IN_FILE
    out_path = FIRST_OUT_PATH
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # 检查训练数据文件是否存在
    if not os.path.exists(train_path):
        print(f"Training file not found for {co}. Skipping...")
        continue

    # 检查预测数据文件是否存在
    if not os.path.exists(file_path):
        print(f"Prediction file not found for {co2}. Skipping...")
        continue

    #判断是否存在同名文件
    if os.path.exists(out_path):
        print(f"File already exists for {co2}. Skipping...")
        continue

    # 读取训练数据
    df = pd.read_csv(train_path)
    df['name'].fillna('', inplace=True)
    df['gender'].fillna('', inplace=True)
    df = df[df.gender.isin(['M', 'F'])]

    X = df.name
    y = df.gender == 'M'

    # 分割训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

    # 训练模型
    pipeline.fit(X_train, y_train)

    # 评估模型
    y_pre = predict_gender(X_test)
    y_pred = y_pre['genders'].values  # this works now
    mask = y_pred != ''
    y_test_filtered = y_test[mask]
    y_pred_filtered = y_pred[mask]
    y_pred_bool = (y_pred_filtered == 'M')
    accuracy = accuracy_score(y_test_filtered, y_pred_bool)
    print(f"Model accuracy: {accuracy}")

    new_row = pd.DataFrame({'样本': [co], '预测': [co2], '准确率': [accuracy]})
    data2 = pd.concat([data2, new_row], ignore_index=True)

    #判断是否存在同名文件
    if os.path.exists(out_path):
        print(f"File already exists for {co2}. Skipping...")
        continue

    # 预测
    dfrd = pd.read_csv(file_path)
    author_array = dfrd['author'].fillna('')
    preds = predict_gender(author_array)
    dfrd['pre gender'] = preds['genders']
    dfrd['proba'] = preds['probability']
    print(dfrd)
    
    # 保存结果
    dfrd.to_csv(out_path, index=False)

    print(f"Prediction saved to {out_path}")
    print("--------------------")

print("All processing completed.")
confidence_path = FIRST_CHECK_OUT
# 检查并创建父目录（如果不存在）
os.makedirs(os.path.dirname(confidence_path), exist_ok=True)
data2.to_csv(confidence_path, index=False)
print(f"Summary results saved {confidence_path}")

In [ ]:
# Merge all outcomes
def merge_csv_files(folder_path):
    # 获取文件夹内所有的CSV文件
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

    # 初始化一个空的DataFrame用于存储合并后的数据
    combined_df = pd.DataFrame()
    
    # 遍历所有CSV文件并合并
    for csv_file in tqdm(csv_files):
        file_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(file_path)
        combined_df = pd.concat([combined_df, df], ignore_index=True)

    return combined_df

In [ ]:
# 示例用法
folder_path = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_first_bycountry_predicted" 
output_file = FIRST_ALL_OUT
combined_df = merge_csv_files(folder_path)
combined_df = combined_df.sort_values(by='index_num')
# 将合并后的数据保存到新的CSV文件中
combined_df.to_csv(output_file, index=False)
print(f"合并完成，结果已保存到 {output_file}")

# Last Corresponding author

In [ ]:

LAST_IN_FILE = PROJECT_ROOT / "data" / "interim" / "publication" / "2_last_bycountry"/ "code_{co}.csv"


LAST_OUT_PATH = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_last_bycountry_predicted" / "pred_{co2}.csv"
LAST_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

LAST_ALL_OUT = PROJECT_ROOT / "data" / "interim" / "publication"/ "2.1_last_NameAndCountry_Predicted.csv"
LAST_ALL_OUT.parent.mkdir(parents=True, exist_ok=True)

LAST_CHECK_OUT = PROJECT_ROOT / "data" / "check" / "gender_classification" / "1.1_last_预测准确率.csv"
LAST_CHECK_OUT.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
warnings.filterwarnings('ignore')
# 定义概率阈值
confidence_level = 0.7

# 读取数据
data = pd.read_excel(
    FILE_PATH)
data2 = pd.DataFrame(columns=['样本', '预测', '准确率'])
data

In [ ]:
# 定义预处理函数
def preprocessor(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub('[^a-zA-Z]', '', x)
    return x.lower()



# 创建向量化器和分类器
vec = CountVectorizer(
    min_df=50,
    ngram_range=(2, 8),
    analyzer='char',
    preprocessor=lambda x: f'^{x.lower().replace(" ", "")},'
)

clf = LogisticRegression(max_iter=1000)
pipeline = Pipeline([('vectorizer', vec), ('logreg', clf)])


def predict_gender(names):
    probas = pipeline.predict_proba(names)
    genders = []
    max_probas = []
    for proba in probas:
        if proba[0] > confidence_level:
            genders.append('F')
            max_probas.append(proba[0])
        elif proba[1] > confidence_level:
            genders.append('M')
            max_probas.append(proba[0])
        else:
            genders.append('')
            max_probas.append(proba[0])
    return pd.DataFrame({'genders': genders, 'probability': max_probas})


In [ ]:
# 循环处理每对样本和预测
for index, row in data.iterrows():    

    #if index>10:
        #continue
        
    co = row['样本']
    co2 = row['预测']

    print(f"Processing: Sample - {co}, Prediction - {co2}")

    train_path = TRAIN_PATH
    file_path = LAST_IN_FILE
    out_path = LAST_OUT_PATH
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # 检查训练数据文件是否存在
    if not os.path.exists(train_path):
        print(f"Training file not found for {co}. Skipping...")
        continue

    # 检查预测数据文件是否存在
    if not os.path.exists(file_path):
        print(f"Prediction file not found for {co2}. Skipping...")
        continue

    #判断是否存在同名文件
    if os.path.exists(out_path):
        print(f"File already exists for {co2}. Skipping...")
        continue

    # 读取训练数据
    df = pd.read_csv(train_path)
    df['name'].fillna('', inplace=True)
    df['gender'].fillna('', inplace=True)
    df = df[df.gender.isin(['M', 'F'])]

    X = df.name
    y = df.gender == 'M'

    # 分割训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

    # 训练模型
    pipeline.fit(X_train, y_train)

    # 评估模型
    y_pre = predict_gender(X_test)
    y_pred = y_pre['genders'].values  # this works now
    mask = y_pred != ''
    y_test_filtered = y_test[mask]
    y_pred_filtered = y_pred[mask]
    y_pred_bool = (y_pred_filtered == 'M')
    accuracy = accuracy_score(y_test_filtered, y_pred_bool)
    print(f"Model accuracy: {accuracy}")

    new_row = pd.DataFrame({'样本': [co], '预测': [co2], '准确率': [accuracy]})
    data2 = pd.concat([data2, new_row], ignore_index=True)

    #判断是否存在同名文件
    if os.path.exists(out_path):
        print(f"File already exists for {co2}. Skipping...")
        continue

    # 预测
    dfrd = pd.read_csv(file_path)
    author_array = dfrd['author'].fillna('')
    preds = predict_gender(author_array)
    dfrd['pre gender'] = preds['genders']
    dfrd['proba'] = preds['probability']
    print(dfrd)
    
    # 保存结果
    dfrd.to_csv(out_path, index=False)

    print(f"Prediction saved to {out_path}")
    print("--------------------")

print("All processing completed.")
confidence_path = LAST_CHECK_OUT
# 检查并创建父目录（如果不存在）
os.makedirs(os.path.dirname(confidence_path), exist_ok=True)
data2.to_csv(confidence_path, index=False)
print(f"Summary results saved {confidence_path}")

In [ ]:
def merge_csv_files(folder_path):
    # 获取文件夹内所有的CSV文件
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

    # 初始化一个空的DataFrame用于存储合并后的数据
    combined_df = pd.DataFrame()
    
    # 遍历所有CSV文件并合并
    for csv_file in tqdm(csv_files):
        file_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(file_path)
        combined_df = pd.concat([combined_df, df], ignore_index=True)

    return combined_df

In [ ]:

folder_path = LAST_OUT_PATH
output_file = LAST_ALL_OUT
combined_df = merge_csv_files(folder_path)
combined_df = combined_df.sort_values(by='index_num')


combined_df.to_csv(output_file, index=False)
print(f"合并完成，结果已保存到 {output_file}")